# Schemas and Data Types: Explicit StructTypes, Nested Types & Schema Enforcement

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 02_schemas_and_data_types.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How to define explicit StructType schemas, parse complex nested types (ArrayType, MapType, StructType), and enforce data contracts.
- **Why does this topic exist?**: Relying on inferSchema causes runtime performance hits and pipeline failures when dirty records arrive.
- **What problem does it solve?**: InferSchema triggers an extra full dataset scan and silently breaks when data types vary across batch files.
- **Where is it used in real-world Data Engineering?**: Enterprise data ingestion frameworks enforce strict schemas to reject corrupted records into dead-letter queues.

---

## 2. Schema Architecture & Data Types

```
StructType (Schema Definition)
  |-- StructField("order_id", IntegerType, nullable=False)
  |-- StructField("customer_id", IntegerType, nullable=True)
  |-- StructField("items", ArrayType(StructType(...)), nullable=True)  <-- Complex Nested Type
  |-- StructField("metadata", MapType(StringType, StringType))        <-- Key-Value Map Type
```

### Why Explicit Schemas Matter in Production:
1. **Zero Scan Overhead**: Spark skips the pre-scan required by `inferSchema=True`.
2. **Type Safety**: Enforces strict contracts (e.g. `order_id` is never accidentally parsed as String).
3. **Deterministic Evolution**: Allows controlled handling of new or missing columns.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Schemas_DataTypes_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Defining Explicit Schemas for Retail Orders

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType, TimestampType
)

orders_schema = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=False),
    StructField("product_id", IntegerType(), nullable=False),
    StructField("order_ts", TimestampType(), nullable=False),
    StructField("quantity", IntegerType(), nullable=False),
    StructField("unit_price", DoubleType(), nullable=False),
    StructField("discount", DoubleType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("channel", StringType(), nullable=True)
])

# Read with explicit schema enforcement
orders_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv("/opt/spark-data/retail/orders.csv")
)

orders_df.printSchema()
print(f"Total verified orders count: {orders_df.count()}")
orders_df.show(3)

## 4. Complex Nested Types: Parsing JSON (`/opt/spark-data/retail/order_events.json`)

Now let us parse deeply nested JSON containing nested StructTypes and ArrayTypes.

In [ ]:
events_raw = spark.read.text("/opt/spark-data/retail/order_events.json")

# Define nested schema matching JSON structure
event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_time", StringType()),
    StructField("order", StructType([
        StructField("order_id", IntegerType()),
        StructField("status", StringType()),
        StructField("channel", StringType()),
        StructField("items", ArrayType(StructType([
            StructField("product_id", IntegerType()),
            StructField("product_name", StringType()),
            StructField("category", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DoubleType())
        ])))
    ])),
    StructField("customer", StructType([
        StructField("customer_id", IntegerType()),
        StructField("segment", StringType()),
        StructField("region", StringType()),
        StructField("country", StringType())
    ]))
])

# Parse JSON strings with explicit schema
events_df = (
    events_raw
    .select(F.from_json(F.col("value"), event_schema).alias("data"))
    .select("data.*")
)

events_df.printSchema()
events_df.select("event_id", "order.order_id", "customer.country").show(3)

---

## Experiment: Exploding Nested Arrays into Relational Rows

- **1. What are we testing?**: Use F.explode() on order.items to flatten nested array elements into individual row records.
- **2. What do we expect?**: Each item in the items array will become a distinct row retaining parent order attributes.

In [ ]:
flattened = (
    events_df
    .select(
        F.col("event_id"),
        F.col("order.order_id").alias("order_id"),
        F.col("customer.customer_id").alias("customer_id"),
        F.explode(F.col("order.items")).alias("item")
    )
    .select(
        "event_id", "order_id", "customer_id",
        "item.product_name", "item.quantity", "item.unit_price"
    )
)

flattened.show(5)
print(f"Original events: {events_df.count()} -> Flattened rows: {flattened.count()}")

**3. What happened?**: Array items were converted to separate rows; row count expanded proportionally.  
**4. Why did it happen?**: explode() unrolls an ArrayType column, replicating parent fields for every nested element.  
**5. Practical Engineering Lesson**: F.explode() is the primary tool in Spark SQL for normalizing semi-structured document payloads.

---

## Practical Exercises

### Exercise 1 (Beginner): Extract Nested Struct Field
**Problem Statement**: Extract customer segment and country directly using dot-notation into new top-level columns.  
**Input / Context**: events_df

In [ ]:
# Exercise 1: Extract nested fields

### Exercise 2 (Intermediate): Handle Corrupted Record Mode with PERMISSIVE
**Problem Statement**: Load /opt/spark-data/quality/orders_dirty.csv using PERMISSIVE mode and columnNameOfCorruptRecord.  
**Input / Context**: /opt/spark-data/quality/orders_dirty.csv

In [ ]:
# Exercise 2: Corrupted record tracking

### Exercise 3 (Advanced): Construct Schema with MapType
**Problem Statement**: Create a DataFrame with a MapType(StringType, DoubleType) and query a specific map key.  
**Input / Context**: F.create_map()

In [ ]:
# Exercise 3: MapType usage

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Extract Nested Struct Field):
```python
res = events_df.select(
    F.col("order.order_id").alias("order_id"),
    F.col("customer.segment").alias("cust_segment"),
    F.col("customer.country").alias("cust_country")
)
res.show(3)
```

#### Solution 2 (Handle Corrupted Record Mode with PERMISSIVE):
```python
dirty_schema = orders_schema.add(StructField("_corrupt_record", StringType(), True))
dirty_df = (
    spark.read
    .option("header", True)
    .option("mode","PERMISSIVE")
    .option("columnNameOfCorruptRecord","_corrupt_record")
    .schema(dirty_schema)
    .csv("/opt/spark-data/quality/orders_dirty.csv")
)
print("Dirty records count with nulls or format errors:")
dirty_df.filter(F.col("_corrupt_record").isNotNull()).select("_corrupt_record").show(3, truncate=False)
```

#### Solution 3 (Construct Schema with MapType):
```python
map_df = spark.createDataFrame([
    (1, {"tax": 12.5,"discount": 5.0}),
    (2, {"tax": 8.0,"discount": 0.0})
], ["id","financials"])
map_df.printSchema()
map_df.select("id", F.col("financials")["tax"].alias("tax_amount")).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Explicit Schemas**: Mandatory in production; eliminates scan overhead and enforces contract safety.
- **Nested Data**: Spark natively supports StructType, ArrayType, and MapType.
- **explode()**: Converts ArrayType elements into individual relational rows.

### Essential Technical Terms
- `StructType`: Collection of StructFields representing a schema.
- `StructField`: Single column definition (name, type, nullable).
- `explode`: Function that flattens an array into multiple rows.

### Common Engineering Mistakes
- **inferSchema=True in Production**: Slows down ingestion and risks schema changes when dirty data arrives.
- **Ignoring Nullability**: Setting nullable=False does not actively reject nulls in CSV; it is a metadata hint.

### Interview Questions on This Topic
- **Q: Why use explicit schemas over inferSchema?**
  - *A*: Explicit schemas avoid an extra full data scan and guarantee type safety across production batches.
- **Q: How do you query nested StructType fields?**
  - *A*: Using dot notation (e.g. col('customer.address.city')).

### What to Study Next
Open 03_dataframe_and_spark_sql/03_dataframe_transformations.ipynb for the complete transformation catalog.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')